# HinEmo — Manual Validation Split
Samples 1,000 random comments from the pre-labeling pool and splits them into
4 files of 250 each, for blind manual relabeling by 4 different people.
Used later to compute Cohen's Kappa against GPT's labels (Step 1.5).

In [8]:
import pandas as pd
import random

SOURCE_PATH = "../data/interim/youtube_codemixed_stage4.csv"
OUTPUT_DIR = "../data/interim/manual_validation"
TOTAL_SAMPLE = 1000
N_FILES = 4
PER_FILE = TOTAL_SAMPLE // N_FILES

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(SOURCE_PATH)
print(f"Loaded {len(df)} total comments")

random.seed(42)
sample_idx = sorted(random.sample(range(len(df)), TOTAL_SAMPLE))
sample = df.iloc[sample_idx][["source_id", "text_clean"]].reset_index(drop=True)
print(f"Sampled {len(sample)} comments for manual validation")

Loaded 43236 total comments
Sampled 1000 comments for manual validation


In [9]:
EMOTION_OPTIONS = "anger / joy / sadness / fear / surprise / disgust / neutral"

for i in range(N_FILES):
    chunk = sample.iloc[i * PER_FILE : (i + 1) * PER_FILE].copy()
    chunk = chunk.rename(columns={"text_clean": "comment_text"})
    chunk.insert(0, "row_number", range(1, len(chunk) + 1))
    chunk["your_emotion_label"] = ""  # they fill this in
    chunk["instructions"] = ""
    chunk.loc[0, "instructions"] = (
        f"Fill in 'your_emotion_label' with ONE of: {EMOTION_OPTIONS}. "
        f"Use 'neutral' if there's no clear emotion. Leave blank only if genuinely unsure."
    )

    filename = f"HinEmo_ManualValidation_Part{i+1}of{N_FILES}_250comments.csv"
    filepath = os.path.join(OUTPUT_DIR, filename)
    chunk.to_csv(filepath, index=False)
    print(f"Saved {filename} ({len(chunk)} rows)")

print(f"\nAll {N_FILES} files saved to {OUTPUT_DIR}")
print("Each person labels 'your_emotion_label' for their assigned file.")
print("Send them back to you, then we compute Cohen's Kappa against GPT's labels.")

Saved HinEmo_ManualValidation_Part1of4_250comments.csv (250 rows)
Saved HinEmo_ManualValidation_Part2of4_250comments.csv (251 rows)
Saved HinEmo_ManualValidation_Part3of4_250comments.csv (251 rows)
Saved HinEmo_ManualValidation_Part4of4_250comments.csv (251 rows)

All 4 files saved to ../data/interim/manual_validation
Each person labels 'your_emotion_label' for their assigned file.
Send them back to you, then we compute Cohen's Kappa against GPT's labels.


In [10]:
all_ids = []
for i in range(N_FILES):
    filename = f"HinEmo_ManualValidation_Part{i+1}of{N_FILES}_250comments.csv"
    filepath = os.path.join(OUTPUT_DIR, filename)
    part_df = pd.read_csv(filepath)
    all_ids.extend(part_df["source_id"].tolist())

print(f"Total IDs across all files: {len(all_ids)}")
print(f"Unique IDs: {len(set(all_ids))}")
print("OK — no overlap" if len(all_ids) == len(set(all_ids)) else "WARNING — overlap detected!")

Total IDs across all files: 1003
Unique IDs: 1001
WARNING — overlap detected!


In [3]:
import pandas as pd

df = pd.read_csv("/Users/harshaggarwal/Projects_4/hinemo_project/data/interim/_checkpoint_labeling.csv")
print(f"Labeled so far: {len(df)}")

before = len(df)
neutral_count = (df["emotion"] == "neutral").sum()
unlabelable_count = df["emotion"].isin(["unknown", "error"]).sum()
df_clean = df[~df["emotion"].isin(["unknown", "error", "neutral"])]

print(f"Dropped {before - len(df_clean)} rows total "
      f"({neutral_count} neutral, {unlabelable_count} unlabelable)")
print(f"\nFinal usable labeled pool: {len(df_clean)}")
print("\nLabel distribution:")
print(df_clean["emotion"].value_counts())

df_clean.to_csv("/Users/harshaggarwal/Projects_4/hinemo_project/data/interim/youtube_labeled_stage5.csv", index=False)
print("\nSaved as youtube_labeled_stage5.csv")

Labeled so far: 41500
Dropped 16085 rows total (10641 neutral, 5444 unlabelable)

Final usable labeled pool: 25415

Label distribution:
emotion
joy         8918
anger       5271
disgust     5053
sadness     3774
fear        1528
surprise     871
Name: count, dtype: int64

Saved as youtube_labeled_stage5.csv


In [4]:
import pandas as pd

df_clean = pd.read_csv("../data/interim/youtube_labeled_stage5.csv")

counts = df_clean["emotion"].value_counts()
percentages = df_clean["emotion"].value_counts(normalize=True) * 100

summary = pd.DataFrame({
    "count": counts,
    "percentage": percentages.round(1)
})

print(f"Total labeled pool: {len(df_clean)}\n")
print(summary)

Total labeled pool: 25415

          count  percentage
emotion                    
joy        8918        35.1
anger      5271        20.7
disgust    5053        19.9
sadness    3774        14.8
fear       1528         6.0
surprise    871         3.4


In [5]:
import pandas as pd

df_clean = pd.read_csv("/Users/harshaggarwal/Projects_4/hinemo_project/data/Extra(fortesting)/emotion_train_final.csv")
counts = df_clean["emotion"].value_counts()
percentages = df_clean["emotion"].value_counts(normalize=True) * 100

summary = pd.DataFrame({
    "count": counts,
    "percentage": percentages.round(1)
})

print(f"Total labeled pool: {len(df_clean)}\n")
print(summary)

Total labeled pool: 5083

         count  percentage
emotion                   
anger     2375        46.7
joy       1990        39.2
sadness    427         8.4
trust      291         5.7
